<h2>
    <b>
        Project วิชา Mathematical for Data Science !     
    </b>
</h2>
created and programming by Arthakorn Narard. Student's ID is 67011578.

<h3>
    <b>Step 1: Load the dataset and look at its structure</b>   
    <b>ตรวจสอบข้อมูลของ Datasets และทำการ Data Manipulating (Clean, Normalization etc.)</b>
</h3>
<h4>
    <b>
        outlines ของ section
    </b>
</h4>
<ul>
    <li>ตรวจสอบข้อมูลของ Datasets (RadioML 2016)</li>
    <li>Clean และ Normalization ข้อมูลเพื่อพร้อมสำหรับการทำ model (จะมีการตัดสินใจภายหลัง)</li>
</ul>

In [ ]:
# ทำการ Import Module หรือ Library ที่จำเป็นสำหรับการอ่านข้อมูล
import pickle               # reads .pkl files (Python's format for saving objects)
import zipfile              # opens .zip files
from pathlib import Path    # "Path" makes working with file locations easier
import numpy as np          # numpy = fast number arrays; "np" is a short nickname

# สร้างตัวแปรสำหรับการเก็บ Path ของ Folder ที่มีการเก็บข้อมูล Datasets เอาไว้
data_folder = Path("../data")   # มีการเก็บเอาไว้ใน folder เดียวกันและสร้างเอาไว้ชื่อ data
data_files_name = "radioml-2016-dataset.zip"

# สร้างตัวแปรสำหรับการระบุ Path ของไฟล์ Datasets 
zip_path = data_folder / data_files_name    # ทำการ join ให้อยู่ในรูปแบบ ../data/radioml-2016-dataset.zip
pkl_path = data_folder / "RML2016.10a_dict.pkl" 

# ทำการแสดงข้อความว่า "พบไฟล์" ถ้าไม่พบให้มีการแสดงคำว่า "ไม่พบไฟล์ กรุณาตรวจสอบไฟล์อีกครั้ง"
if(zip_path.exists()==True):
    print("Zip file was found !")
else: 
    print("Zip file doesn't found, Please check your files.")

Zip file was found !


In [17]:
# ขั้นตอนสำหรับการแตกไฟล์ Extract Files เพื่อพร้อมสำหรับการอ่านข้อมูล
# คำสั่ง if not จะมีลักษณะการทำงานคล้าย if-else แต่ว่าจะตรงกันข้าม
if not pkl_path.exists():   # ตรวจสอบว่ามีไฟล์ pkl ไหม ? ให้ทำคำสั่ง else ; ทำคำสั่งด้านล่าง
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_folder)      # ทำการ Unzip ไฟล์เพื่อให้ได้เป็นไฟล์นามสกุล .pkl ออกมา 
    print("Extracted the file.")
else:
    print("Already extracted, skipping.")

# ทำการแสดงผลขนาดของไฟล์ Datasets ออกมาเป็นหน่วย GB
size = pkl_path.stat().st_size  # สร้างตัวแปรสำหรับการเก็บขนาดของ file ซึ่งตรวจสอบออกมาแล้วว่ามีจำนวนเท่ากับ 9 ตัวอักษรตีความว่าจะมีขนาดประมาณ GB
if(len(str(size))>=9):
    print(f"Size : {round(size/1e9,2)} GB") # ถ้าจำนวนตัวอักษรมากกว่าเท่ากับ 9 จะให้มีการแสดงออกมาเป็นหน่วย GB
else:
    print(f"Size : {round(size/1e6,1)} MB") # ถ้าไม่ใช่แสดงว่าน้อยกว่า 9 จะให้มีการแสดงออกมาเป็นหน่วย MB

Already extracted, skipping.
Size : 0.64 GB


In [18]:
# Cell 3: Load the file safely

# Danger: a normal pickle.load() runs ANY function named inside the file.
# A "class" is a blueprint. We copy Python's normal loader (pickle.Unpickler)
# and change one part: find_class, which is called each time the file asks for
# a function. We allow only the numpy functions that rebuild arrays.
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # The allow list: (module name, function name) pairs we trust.
        allowed = [
            ("numpy", "dtype"),                         # describes the number type
            ("numpy", "ndarray"),                       # the array object itself
            ("numpy.core.multiarray", "_reconstruct"),  # rebuilds an array
        ]
        if (module, name) in allowed:
            return super().find_class(module, name)     # allowed: do the normal thing
        # Anything else: stop loading immediately with an error.
        raise pickle.UnpicklingError(f"Blocked unsafe item: {module}.{name}")

# "rb" = read binary (the file holds raw bytes, not text).
with open(pkl_path, "rb") as f:
    # encoding="latin1" is needed because the file was made with Python 2 (D2).
    data = SafeUnpickler(f, encoding="latin1").load()

print("Loaded safely!")

Loaded safely!


In [19]:
# Cell 4: Look at the structure

# 1) What kind of object is it, and how many items does it hold?
print("Type:", type(data))
print("Number of keys:", len(data))

# 2) A dictionary maps keys to values. list(...) turns the keys into a list.
keys = list(data.keys())
print("First 3 keys:", keys[:3])   # [:3] = the first 3 items

# 3) Each key is a pair (modulation, snr). k[0] is the first part, k[1] the second.
#    {... for k in keys} builds a "set", which keeps each value once (no repeats).
#    sorted() puts them in order.
modulations = sorted({k[0] for k in keys})
snrs = sorted({k[1] for k in keys})
print("Modulations:", len(modulations), modulations)
print("SNR levels:", len(snrs), snrs)

# 4) Look inside one value: the data stored under the first key.
one = data[keys[0]]
print("Value type:", type(one))
print("Shape:", one.shape)     # size of each dimension
print("Number type:", one.dtype)

# 5) Show the first signal, first 5 numbers of each row.
#    [0, :, :5] means: item 0, all rows (:), columns 0 to 4 (:5).
print(one[0, :, :5])

Type: <class 'dict'>
Number of keys: 220
First 3 keys: [('QPSK', 2), ('PAM4', 8), ('AM-DSB', -4)]
Modulations: 11 ['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK', 'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']
SNR levels: 20 [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
Value type: <class 'numpy.ndarray'>
Shape: (1000, 2, 128)
Number type: float32
[[-0.00590147 -0.00234582 -0.00074506 -0.00534572 -0.00578942]
 [-0.00779554 -0.00781637 -0.00401967 -0.00511351 -0.00593952]]
